In [ ]:
import os, re, json, gc, time, platform, warnings, unicodedata
from datetime import datetime
from typing import List, Tuple, Dict, Any

import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from docx import Document
from striprtf.striprtf import rtf_to_text

# =========================
# KONFIGURÁCIA
# =========================
MODE = 'title'        # 'title' alebo 'keyword' (primárne 'title')

INPUT_DIR   = './Input'
OUTPUT_DIR  = './Output'
THESAURUS_XLSX = './Thesaurus/SK_Local_Thesaurus.xlsx'
THESAURUS_COL  = 'slovak_terms'
STOPWORDS_TXT  = './Input/stopword_dictionary.txt'

# BENCHMARK
MAX_BENCH_SECTIONS = 8          # koľko sekcií otestujeme (random/first)
PROPOSALS_PER_PROMPT = (3, 5)   # generuj 3-5 návrhov
DISABLE_THRESHOLD_SCORE = 0.55  # modely s priemerným skóre < prah odporučíme vypnúť

# GENERÁCIA
MAX_NEW_TOKENS = 256
DO_SAMPLE      = True
TEMPERATURE    = 0.6
TOP_P          = 0.95
REPETITION_PENALTY = 1.3
MAX_TEXT_CHARS = 2000

# 4-bit kvantizácia (globálny prepínač pre menšie VRAM; môžeš zrušiť na silnej GPU)
FORCE_4BIT_PRIMARY = os.environ.get("FORCE_4BIT", "false").lower() == "true"

# MODELY (poradie je poradie benchmarku aj default nasadenia)
ENABLED_MODELS = [
    "Qwen/Qwen2.5-3B-Instruct",
    "google/gemma-2-2b-it",
]

# voliteľné dodatočné vypnutia cez súbor
DISABLED_FILE = "./disabled_models.txt"

# OS-aware alokátor (Windows ignoruje expandable_segments)
def _configure_cuda_allocator():
    parts = ["max_split_size_mb:128"]
    if platform.system() == "Linux":
        parts.insert(0, "expandable_segments:True")
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)

warnings.filterwarnings("ignore", message=r".*expandable_segments not supported.*")
_configure_cuda_allocator()
os.environ.setdefault("PYTORCH_FLASH_ATTENTION", "0")

# =========================
# STOPWORDS
# =========================
def load_stopwords(path=STOPWORDS_TXT):
    try:
        with open(path, 'r', encoding='utf-8') as f:
            return {line.strip().lower() for line in f if line.strip()}
    except FileNotFoundError:
        print(f'[INFO] No stopword file at {path}. Continuing without.')
        return set()
    except Exception as e:
        print(f'[WARN] Failed to load stopwords: {e}')
        return set()
STOPWORDS = load_stopwords()

# =========================
# TEZAURUS
# =========================
def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)
    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))
    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]
    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL) -> List[str]:
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f'Thesaurus file not found: {xlsx_path}')
    df = pd.read_excel(xlsx_path, engine='openpyxl')
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')
    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))
    seen, terms = set(), []
    for t in parts:
        t = t.strip()
        if not t: continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS: continue
        if t not in seen:
            seen.add(t); terms.append(t)
    print(f'[INFO] Thesaurus terms loaded: {len(terms)}')
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

def terms_matched_in_text(text: str, max_terms: int = 30) -> List[str]:
    hits = {}
    if not text: return []
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0:
            hits[canon] = cnt
    ordered = [k for k,_ in sorted(hits.items(), key=lambda kv: (-kv[1], -len(kv[0])))]
    return ordered[:max_terms]

# =========================
# DOKUMENTY: čítanie & sekcie
# =========================
_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path: str) -> List[Tuple[str, str]]:
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'[WARN] RTF split failed {path}: {e}')
        return []

# =========================
# LLM LOADER (viac modelov + 4-bit kde dáva zmysel)
# =========================
def load_llm(model_name: str):
    device_map = {"": "cuda:0"} if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    use_4bit = False
    qconf = None
    # malý common-sense: ak je model <= ~3B a máme málo VRAM, 4-bit pomôže;
    # TinyLlama dáme 4-bit na istotu; iné necháme default BF16/FP16 ak nevyžadujeme FORCE_4BIT_PRIMARY
    if model_name.endswith("TinyLlama-1.1B-Chat-v1.0"):
        use_4bit = True
    elif FORCE_4BIT_PRIMARY and ("Phi-3" in model_name or "Qwen2.5-3B" in model_name or "gemma-2-2b" in model_name or "Llama-3.2-3B" in model_name):
        use_4bit = True

    if use_4bit:
        qconf = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
        )

    kwargs = dict(
        device_map=device_map,
        torch_dtype=dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    if qconf is not None:
        kwargs["quantization_config"] = qconf

    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    pipe = pipeline(
                    "text-generation",
                    model=model,
                    tokenizer=tok,
                    device_map=device_map,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=DO_SAMPLE,
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    return_full_text=False,
                    repetition_penalty=REPETITION_PENALTY)
    
    return pipe, model, tok, f"{model_name}{' (4bit)' if use_4bit else ''}"

def unload_llm(model, tok, pipe):
    try:
        del model, tok, pipe
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# =========================
# PROMPTY + PARSING
# =========================
def truncate_text(s: str, max_chars: int = MAX_TEXT_CHARS) -> str:
    s = (s or "").strip()
    if len(s) <= max_chars: return s
    return s[: max_chars//2] + "\n...\n" + s[- max_chars//2 :]

def prompt_titles(text: str, th_priority: List[str], th_sample: List[str], n_min=3, n_max=5) -> str:
    pri = "\n".join(th_priority[:20]) if th_priority else "(žiadne priame zhody)"
    samp = "\n".join(th_sample[:40])
    return (
        f"ÚLOHA: Navrhni {n_min}–{n_max} presných a zrozumiteľných PRÁVNYCH NADPISOV (3–12 slov) k textu nižšie.\n"
        "Požiadavky: vecné, bez bodky na konci, žiadne úvodzovky.\n"
        "Výstup: Vráť IBA čistý JSON zoznam reťazcov, napr. [\"Nadpis A\",\"Nadpis B\"]. ŽIADNY komentár.\n\n"
        f"TEZAURUS – PRIO (zhody v texte):\n{pri}\n\n"
        f"TEZAURUS – VÝBER:\n{samp}\n\n"
        f"TEXT:\n{text}\n\n"
        "ODPOVEĎ:"
    )

def parse_titles(raw: str) -> List[str]:
    # pokus o čistý JSON
    s = raw.strip()
    try:
        start = s.index('['); end = s.rindex(']'); blob = s[start:end+1]
        js = json.loads(blob)
        out = [str(x).strip() for x in js if isinstance(x, (str,))]
        out = [re.sub(r'[\.，。…]+$','', o).strip(' "„”') for o in out if o]
        return [o for o in out if len(o.split())>=3 and len(o.split())<=12]
    except Exception:
        # fallback: každý riadok ako kandidat
        lines = [re.sub(r'^[\-\d\.\)\s]+','', ln).strip() for ln in s.splitlines() if ln.strip()]
        out = [re.sub(r'[\.，。…]+$','', ln).strip(' "„”') for ln in lines]
        return [o for o in out if 3 <= len(o.split()) <= 12]

# =========================
# HEURISTICKÝ SKÓROVAČ (bez embeddings)
# =========================
def simple_score(title: str, src_text: str, priority_terms: List[str]) -> float:
    """0..1 – mix: dĺžka v slovách, pokrytie tezauru v texte, penalizácia 'prázdnych' slov."""
    if not title: return 0.0
    words = title.split()
    n = len(words)

    # dĺžkový profil (sweet spot 4–10 slov)
    if 4 <= n <= 10: len_score = 1.0
    else: len_score = max(0.0, 1.0 - abs(n - 7) * 0.12)

    t_na = strip_accents(title.lower())
    # pokrytie prior. termínov (ak nejaké sú)
    cov = 0.0
    if priority_terms:
        hits = 0
        for term in priority_terms[:20]:
            term_na = strip_accents(term.lower())
            if term_na in t_na:
                hits += 1
        cov = hits / max(1, min(10, len(priority_terms)))

    # penalizácia 'prázdnych' tokenov (stopwords-only nadpisy)
    non_stop = [w for w in re.findall(r'\w+', t_na) if w not in STOPWORDS]
    density = len(non_stop) / max(1, len(re.findall(r'\w+', t_na)))
    dens_score = 0.3 + 0.7 * density  # nikdy nie 0

    # slabá sankcia za interpunkciu/upper
    punc_penalty = 0.0
    if title.strip().endswith(('.', '…')): punc_penalty += 0.1
    if sum(ch.isupper() for ch in title) > 0.7*sum(ch.isalpha() for ch in title if ch.isalpha()):
        punc_penalty += 0.1

    score = 0.5*len_score + 0.4*cov + 0.2*dens_score - punc_penalty
    return max(0.0, min(1.0, score))

def pick_best(titles: List[str], src_text: str, priority_terms: List[str]) -> Dict[str, Any]:
    scored = [(t, simple_score(t, src_text, priority_terms)) for t in titles]
    scored.sort(key=lambda x: x[1], reverse=True)
    return {
        "winner": scored[0][0] if scored else "",
        "winner_score": scored[0][1] if scored else 0.0,
        "scored": scored[:]
    }

# =========================
# BENCHMARK + DRIVER
# =========================
def truncate_for_model(text: str) -> str:
    s = (text or "").strip()
    if len(s) <= MAX_TEXT_CHARS: return s
    return s[: MAX_TEXT_CHARS//2] + "\n...\n" + s[- MAX_TEXT_CHARS//2 :]

def collect_sections(input_dir=INPUT_DIR, limit=MAX_BENCH_SECTIONS) -> List[Tuple[str,str,str]]:
    """return list of (file, header, text) up to limit sections."""
    out = []
    files = sorted(os.listdir(input_dir), key=str.lower)
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue
        for header, text in secs:
            if text and text.strip():
                out.append((f, header, text))
                if len(out) >= limit:
                    return out
    return out

def run_benchmark_on_model(model_name: str, bench_sections: List[Tuple[str,str,str]]) -> Dict[str, Any]:
    print(f"\n[MODEL] Loading {model_name} ...")
    t0_load = time.perf_counter()
    try:
        pipe, model, tok, label = load_llm(model_name)
    except Exception as e:
        print(f"[ERROR] Failed to load {model_name}: {e}")
        return {"model": model_name, "label": model_name, "ok": False, "error": str(e)}
    load_time = time.perf_counter() - t0_load
    print(f"[MODEL] Loaded {label} in {load_time:.2f}s")

    rows = []
    total_time = 0.0
    ok = True
    for (f, header, text) in bench_sections:
        short = truncate_for_model(text)
        prio = terms_matched_in_text(text, max_terms=30)
        sample = prio if prio else CANON_TERMS[:40]

        prompt = prompt_titles(short, prio, sample, PROPOSALS_PER_PROMPT[0], PROPOSALS_PER_PROMPT[1])
        t1 = time.perf_counter()
        try:
            out = pipe(prompt)
            raw = out[0]["generated_text"] if isinstance(out, list) else str(out)
            titles = parse_titles(raw)
            pick = pick_best(titles, short, prio)
            dt = time.perf_counter() - t1
            total_time += dt
            rows.append({
                "file": f,
                "header": header,
                "n_proposals": len(titles),
                "winner": pick["winner"],
                "winner_score": round(pick["winner_score"], 3),
                "avg_title_len": round(sum(len(t.split()) for t in titles)/max(1,len(titles)), 2) if titles else 0,
                "latency_s": round(dt, 3),
            })
        except Exception as e:
            ok = False
            rows.append({
                "file": f, "header": header, "n_proposals": 0,
                "winner": "", "winner_score": 0.0, "avg_title_len": 0, "latency_s": None,
                "error": str(e)
            })

    # sumarizácia
    avg_score = sum(r["winner_score"] for r in rows)/max(1,len(rows))
    avg_latency = sum(r["latency_s"] for r in rows if r["latency_s"] is not None)/max(1, len([r for r in rows if r["latency_s"] is not None]))
    result = {
        "model": model_name,
        "label": label,
        "ok": ok,
        "load_time_s": round(load_time, 3),
        "avg_winner_score": round(avg_score, 3),
        "avg_latency_s": round(avg_latency, 3),
        "records": rows,
    }
    unload_llm(model, tok, pipe)
    return result

def apply_disabled_list(models: List[str]) -> List[str]:
    disabled = set()
    if os.path.exists(DISABLED_FILE):
        try:
            with open(DISABLED_FILE, 'r', encoding='utf-8') as f:
                for ln in f:
                    ln = ln.strip()
                    if ln and not ln.startswith('#'):
                        disabled.add(ln)
        except Exception as e:
            print(f"[WARN] Couldn't read {DISABLED_FILE}: {e}")
    enabled = [m for m in models if m not in disabled]
    if disabled:
        print(f"[INFO] Disabled by file: {', '.join(sorted(disabled))}")
    return enabled

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    bench_sections = collect_sections(INPUT_DIR, MAX_BENCH_SECTIONS)
    if not bench_sections:
        print("[ERROR] No sections found to benchmark.")
        return

    models_to_run = apply_disabled_list(ENABLED_MODELS)
    all_results = []
    flat_rows = []

    for m in models_to_run:
        res = run_benchmark_on_model(m, bench_sections)
        all_results.append(res)
        # flatten
        for r in res.get("records", []):
            r2 = r.copy()
            r2["model"] = res["label"]
            flat_rows.append(r2)

    # Ulož výsledky
    today = datetime.today().strftime("%Y-%m-%d")
    df = pd.DataFrame(flat_rows, columns=[
        "model","file","header","n_proposals","winner","winner_score","avg_title_len","latency_s"
    ])
    xlsx_path = os.path.join(OUTPUT_DIR, f"slm_benchmark_{today}.xlsx")
    csv_path  = os.path.join(OUTPUT_DIR, f"slm_benchmark_{today}.csv")
    try: df.to_excel(xlsx_path, index=False, engine="openpyxl"); print(f"[SAVED] {xlsx_path}")
    except Exception as e: print(f"[WARN] Excel write failed: {e}")
    try: df.to_csv(csv_path, index=False, encoding="utf-8-sig"); print(f"[SAVED] {csv_path}")
    except Exception as e: print(f"[WARN] CSV write failed: {e}")

    # Sumarizačný výpis + návrh na vypnutie
    summary_rows = []
    for res in all_results:
        summary_rows.append({
            "model": res["label"],
            "ok": res["ok"],
            "load_time_s": res.get("load_time_s"),
            "avg_winner_score": res.get("avg_winner_score"),
            "avg_latency_s": res.get("avg_latency_s")
        })
    sdf = pd.DataFrame(summary_rows).sort_values(
        by=["avg_winner_score","avg_latency_s"], ascending=[False, True]
    )
    print("\n===== BENCHMARK SUMMARY =====")
    print(sdf.to_string(index=False))

    # navrh na vypnutie
    suggested_disable = sdf[sdf["avg_winner_score"] < DISABLE_THRESHOLD_SCORE]["model"].tolist()
    if suggested_disable:
        print("\n[Suggestion] Consider disabling these (low quality):")
        for m in suggested_disable:
            print(" -", m)
        txt_path = os.path.join(OUTPUT_DIR, f"suggest_disable_{today}.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("# Models suggested to disable (avg_winner_score below threshold)\n")
            f.write(f"# threshold = {DISABLE_THRESHOLD_SCORE}\n\n")
            for m in suggested_disable:
                f.write(m + "\n")
        print(f"[SAVED] {txt_path}")
        print("\n[Manual switch-off] Add models above to disabled_models.txt or remove from ENABLED_MODELS in the script.")
    else:
        print("\n[Suggestion] No model fell below the quality threshold.")

if __name__ == "__main__":
    main()


[INFO] Thesaurus terms loaded: 3000

[MODEL] Loading microsoft/Phi-3-mini-4k-instruct ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

c:\Users\nyx3t\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nyx3t\.cache\huggingface\hub\models--microsoft--Phi-3-mini-4k-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Device set to use cuda:0


[MODEL] Loaded TinyLlama/TinyLlama-1.1B-Chat-v1.0 (4bit) in 39.31s


Token indices sequence length is longer than the specified maximum sequence length for this model (2262 > 2048). Running this sequence through the model will result in indexing errors


[SAVED] ./Output\slm_benchmark_2025-10-14.xlsx
[SAVED] ./Output\slm_benchmark_2025-10-14.csv

===== BENCHMARK SUMMARY =====
                                    model    ok  load_time_s  avg_winner_score  avg_latency_s
                 Qwen/Qwen2.5-3B-Instruct  True       94.704             0.520         19.610
                     google/gemma-2-2b-it  True       82.597             0.412          2.151
TinyLlama/TinyLlama-1.1B-Chat-v1.0 (4bit) False       39.306             0.172         15.825
         microsoft/Phi-3-mini-4k-instruct False      113.472             0.000          0.000
 NousResearch/Hermes-2-Theta-Llama-3.2-3B False          NaN               NaN            NaN

[Suggestion] Consider disabling these (low quality):
 - Qwen/Qwen2.5-3B-Instruct
 - google/gemma-2-2b-it
 - TinyLlama/TinyLlama-1.1B-Chat-v1.0 (4bit)
 - microsoft/Phi-3-mini-4k-instruct
[SAVED] ./Output\suggest_disable_2025-10-14.txt

[Manual switch-off] Add models above to disabled_models.txt or remove from E